In [1]:
from typing import TypedDict,Literal 
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_ollama import ChatOllama
from pydantic import BaseModel , Field
from dotenv import load_dotenv
import os 
import requests
from langchain_mcp_adapters.client import MultiServerMCPClient
import asyncio
import json
load_dotenv()

api_key = os.getenv("USDA_API_KEY")

class AgentState(TypedDict):
    user_query : str
    budget_amount: float | None
    protein_target: float | None
    protein_consumed: float | None
    protein_deficit: float | None
    recommendation : str
    approval:bool
    search_results: list[dict]
    ranked_items: list[dict]
    follow_up_question: str | None
   
llm = ChatOllama(model = "qwen2.5:1.5b")



In [2]:
class extract(BaseModel):
    amount : float|None  = Field(description ="extract the amount from the user input default value is none")
    protein_target : float|None = Field(description = "extract the target protein from the user input default value is none")
    protein_consumed : float|None = Field(description = "extract the consumed protein from the user query default value is none")

system_prompt = """
You are a constraint extraction AI.

Extract the following fields from the user's query:

1. amount
   - Budget amount mentioned by the user.

2. protein_target
   - Desired protein intake in grams.

3. protein_consumed
   - Protein already consumed by the user in grams.

Rules:
- Extract only information explicitly stated by the user.
- Never infer, estimate, or guess values.
- If a field is not mentioned, return None.
- If the query contains no relevant information, return None for all fields.
- Do not perform calculations.
- Do not assume that every number refers to every field.

Examples:

User: "I have ₹300"
amount = 300
protein_target = None
protein_consumed = None

User: "Need 120g protein"
amount = None
protein_target = 120
protein_consumed = None

User: "I've already consumed 40g protein"
amount = None
protein_target = None
protein_consumed = 40

User: "Recommend food"
amount = None
protein_target = None
protein_consumed = None
"""

llm_with_structure = llm.with_structured_output(extract)


def constraint_extract(state:AgentState):
    query = state["user_query"]
    response : extract = llm_with_structure.invoke([SystemMessage(content = system_prompt) , HumanMessage(content = query)])
    
    return {"budget_amount" : response.amount,
            "protein_target" : response.protein_target,
            "protein_consumed" : response.protein_consumed}


In [3]:

class followup(BaseModel):

    question : str

system_prompt2 = """
You are a follow-up question generation assistant.

Your task is to generate a natural question asking the user only for the missing constraints.

Rules:
- Ask only for the constraints provided.
- Do not ask for constraints that are already available.
- If multiple constraints are missing, combine them into a single concise question.
- Keep the question short and natural.
- Do not explain your reasoning.
- Do not mention constraint names literally unless they are user-friendly.
- Return only the question.
"""

def Missing_constraint(state:AgentState):
    missing = []

    if state["budget_amount"] is None:
        missing.append("budget")

    if state["protein_target"] is None:
        missing.append("protein_target")
    if not missing:
        return {
            "missing_constraints": [],
            "follow_up_question": None
        }
    
    llm_with_structure2 = llm.with_structured_output(followup)
    
    response: followup = llm_with_structure2.invoke(
        [
            SystemMessage(content=system_prompt2),
               HumanMessage(
    content=f"""
    The user has not provided the following information:

    {', '.join(missing)}

    Generate a follow-up question asking only for this information.
"""
)
            ])
        

    return {
        "follow_up_question": response.question
    }


In [4]:
client = MultiServerMCPClient(
{
    "swiggy": {        
        "transport": "streamable_http",
        "url": "http://127.0.0.1:8001/mcp"
    }
}
)
tools = await client.get_tools()

  + Exception Group Traceback (most recent call last):
  |   File "c:\swiggy_nutrition_agent\env2\Lib\site-packages\IPython\core\interactiveshell.py", line 3746, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\sivam\AppData\Local\Temp\ipykernel_8980\740936700.py", line 9, in <module>
  |     tools = await client.get_tools()
  |             ^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "c:\swiggy_nutrition_agent\env2\Lib\site-packages\langchain_mcp_adapters\client.py", line 209, in get_tools
  |     tools_list = await asyncio.gather(*load_mcp_tool_tasks)
  |                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "c:\swiggy_nutrition_agent\env2\Lib\site-packages\langchain_mcp_adapters\tools.py", line 590, in load_mcp_tools
  |     async with create_session(
  |                ~~~~~~~~~~~~~~^
  |         connection, mcp_callbacks=mcp_callbacks
  |         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |     ) as tool_session:
  |     ^
  |

In [ ]:
get_addresses = next(t for t in tools if t.name == "get_addresses")
get_restaurant = next(t for t in tools if t.name == "search_restaurants")
get_menu = next(t for t in tools if t.name == "get_menu")

In [ ]:
async def food_search(state:AgentState):
    budget = state["budget_amount"]
    query = state["user_query"]
    addresses_response = await get_addresses.ainvoke({})

    addresses = json.loads(addresses_response[0]["text"])
    question = "Which address would you like to use?"
    for i , address in enumerate(addresses , start = 1):
          question+=f"\n{i}. {address['label']}"
    address_id = None
    for address in addresses:
        if address["label"].lower() == query.lower():
            address_id = address["address_id"]
            break
    
    restaurants = await get_restaurant.ainvoke({"query": "",
                                                "address_id" : address_id})
       
    if address_id is None:
        return {
        "follow_up_question": question
        }   
    for i , restaurants in enumerate(restaurants , start = 1):
        res_id = restaurants["restaurant_id"]
    get_menu.ainvoke({
        "restaurant_id" : res_id
    })


In [ ]:
tools

[StructuredTool(name='get_addresses', description='Get saved delivery addresses for the user.', args_schema={'additionalProperties': False, 'properties': {}, 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x00000243293A7380>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000024329599800>),
 StructuredTool(name='search_restaurants', description="Search restaurants near the given address.\n'query' can be a cuisine/keyword filter (e.g. 'protein', 'budget', 'healthy').", args_schema={'additionalProperties': False, 'properties': {'query': {'default': '', 'type': 'string'}, 'address_id': {'default': 'a1', 'type': 'string'}}, 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x00000243293A7380>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langc

In [ ]:
restaurants = await get_restaurant.ainvoke({
    "query": "",
    "address_id": "a1"
})

print(restaurants)

[{'type': 'text', 'text': '[{"restaurant_id":"r1","name":"Green Bowl Kitchen","cuisine":"Healthy, Salads, Bowls","rating":4.5,"price_for_two":350,"eta_minutes":28},{"restaurant_id":"r2","name":"Protein Punch","cuisine":"High-Protein, Grilled, North Indian","rating":4.3,"price_for_two":450,"eta_minutes":35},{"restaurant_id":"r3","name":"Budget Bites","cuisine":"North Indian, Thali, Combo Meals","rating":4.0,"price_for_two":200,"eta_minutes":25}]', 'id': 'lc_69241866-0b4a-4a0f-b1c8-6a0acefd7526'}]


In [ ]:
get_menu

StructuredTool(name='get_menu', description='Get menu items (with price, protein_g, calories) for a restaurant_id.', args_schema={'additionalProperties': False, 'properties': {'restaurant_id': {'type': 'string'}}, 'required': ['restaurant_id'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x00000243293A7380>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000024329651620>)

In [ ]:
addresses_response = await get_addresses.ainvoke({})
addresses_response

[{'type': 'text',
  'text': '[{"address_id":"a1","label":"Home","line":"123 MG Road, Bengaluru"},{"address_id":"a2","label":"Work","line":"45 Tech Park, Whitefield"}]',
  'id': 'lc_7cf851ad-7d10-4bb1-a1fd-142aaa25feb0'}]